# Chapter 16 — TabPFN Family

Reproduces:
- Figure 16.1: McCarter 2024 reverse-engineering finding — per-layer / per-head
  symmetric energy fraction $\alpha_S$ and skew-symmetric energy fraction
  $\alpha_A$ of TabPFN-lite trained on a small SCM prior.
- Figure 16.2: Cosine recovery of the post-hoc symmetric component to a known
  oracle Gram on a held-out task.
- Table 16.1: Per-layer energy split (printed; LaTeX cell renders the same in
  the chapter).


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tabkernels.architectures import TabPFNLite
from tabkernels.priors import SCMPrior, SCMConfig
from tabkernels.training import PFNTrainer
from tabkernels.transparency import (
    decompose_attention, kernel_energy_split, recovery_cosine,
)

torch.manual_seed(0); np.random.seed(0)
_p = os.getcwd()
while _p and not os.path.isdir(os.path.join(_p, 'affinity', 'book')):
    _p = os.path.dirname(_p)
FIGURES_DIR = os.path.join(_p, 'affinity', 'book', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


## Train TabPFN-lite on a small SCM prior

Two encoder layers, $d_\text{model} = 64$, four heads. SCM prior with
MLP structural equations (so the labelling function is non-linear), $1500$
training steps, Adam $3 \times 10^{-3}$.


In [ ]:
D = 4
prior = SCMPrior(SCMConfig(structural='mlp', edge_prob=0.5, noise_scale=0.3,
                           mlp_hidden=12))
torch.manual_seed(0)
model = TabPFNLite(d_in=D, d_model=64, n_heads=4, n_layers=2, dim_ff=128)
trainer = PFNTrainer(prior=prior, model=model, n_steps=1500,
                     n_ctx=48, n_query=24, d=D, lr=3e-3,
                     eval_every=100, seed=1)
state = trainer.train()
print(f'final train loss : {state.losses[-1]:.4f}')
print(f'final eval  loss : {state.eval_losses[-1]:.4f}')


## Held-out ICL inference

Training is done; deployment is a single forward pass per held-out task.


In [ ]:
model.eval()
with torch.no_grad():
    mses = []
    for k in range(20):
        X_c, y_c, X_q, y_q = prior.sample_episode(48, 24, D, seed=20000 + k)
        yhat = model(X_q, X_c, y_c)
        mses.append((((yhat - y_q) ** 2).mean() / y_q.var().clamp_min(1e-3)).item())
print(f'held-out normalised MSE: {np.mean(mses):.3f} +/- {np.std(mses) / np.sqrt(len(mses)):.3f}')


## Figure 16.1 — McCarter's reverse-engineering finding

McCarter (2024) reports that TabPFN, despite being parameterised with separate
$W_Q$ and $W_K$ (which permits asymmetric similarity scores), learns weights
such that $B = W_Q^\top W_K$ has high symmetric energy $\alpha_S$ and low
skew-symmetric energy $\alpha_A$. The interpretation is that TabPFN's softmax
attention is *effectively* a symmetric kernel smoother, despite its asymmetric
parameterisation. We replicate this on TabPFN-lite by extracting $B$ for each
head of each encoder layer.


In [ ]:
def per_layer_energy_split(model):
    rows = []
    for li, block in enumerate(model.attention_blocks()):
        d = decompose_attention(block)
        for h in range(d['B'].shape[0]):
            a_s, a_a = kernel_energy_split(d['B'][h])
            rows.append((li, h, a_s, a_a))
    return rows


rows = per_layer_energy_split(model)
print('Layer  Head  alpha_S   alpha_A')
for li, h, a_s, a_a in rows:
    print(f'{li:>5d}  {h:>4d}  {a_s:>7.3f}  {a_a:>7.3f}')

mean_s = float(np.mean([r[2] for r in rows]))
mean_a = float(np.mean([r[3] for r in rows]))
print(f'\nmean alpha_S : {mean_s:.3f}')
print(f'mean alpha_A : {mean_a:.3f}')

fig, ax = plt.subplots(1, 1, figsize=(7, 3.8))
xs = np.arange(len(rows))
labels = [f'L{li}H{h}' for li, h, _, _ in rows]
sym = [r[2] for r in rows]; asym = [r[3] for r in rows]
ax.bar(xs, sym, color='C0', label=r'$\alpha_S$')
ax.bar(xs, asym, bottom=sym, color='C3', label=r'$\alpha_A$')
ax.axhline(0.5, color='gray', ls='--', alpha=0.5, label='symmetric / asymmetric parity')
ax.set_xticks(xs); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_ylabel('energy fraction'); ax.set_ylim(0, 1.05)
ax.set_title('Figure 16.1: TabPFN-lite per-head energy split (replicates McCarter 2024)')
ax.legend(loc='upper right')
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_16_01_mccarter_replication.pdf', bbox_inches='tight')
plt.show()


## Figure 16.2 — Cosine recovery to oracle Gram

The post-hoc decomposition diagnostic of Chapter 13 also exposes a *recovery*
metric: the cosine between the *learned* sym/skew kernel matrices on a
held-out batch and a *target* (here, the symmetric oracle Gram constructed
from the SCM ground-truth). High symmetric cosine confirms that the learned
attention's symmetric projection is doing useful work.


In [ ]:
# Pick one held-out task; build oracle Gram from the ground-truth structural
# weights. Since SCM prior emits structural functions per-task and we don't
# expose them directly, we use a proxy: the kernel matrix on the *true* labels
# (smooth functions of x) — which is the same kernel a Bayes-optimal smoother
# would converge to on this task.
torch.manual_seed(7)
X_c, y_c, X_q, y_q = prior.sample_episode(48, 24, D, seed=7777)
X_all = torch.cat([X_c, X_q], dim=0)
y_all = torch.cat([y_c, y_q], dim=0)

# Oracle Gram: outer product of the true labels (rank-1; positive diagonal).
# This is the simplest symmetric target.
oracle_gram_S = (y_all[:, None] * y_all[None, :]).numpy()


def measured_gram_at_layer(model, X_all, y_c, n_ctx, layer_idx):
    """Return the per-pair attention scores (averaged over heads) at the
    given layer's input. We extract by running the tokeniser + first
    `layer_idx` layers, then computing scores on the next attention block's input."""
    model.eval()
    with torch.no_grad():
        seq = model.tokenizer(X_all[:n_ctx].unsqueeze(0), y_c.unsqueeze(0),
                              X_all[n_ctx:].unsqueeze(0))
        for li in range(layer_idx):
            seq = model.layers[li](seq)
        x = model.layers[layer_idx].norm1(seq)
        block = model.layers[layer_idx].attn
        B, N, _ = x.shape
        H, dh = block.n_heads, block.d_h
        q = block.W_Q(x).view(B, N, H, dh)
        k = block.W_K(x).view(B, N, H, dh)
        S = torch.einsum('bnhd,bmhd->bhnm', q, k) / (dh ** 0.5)
        # Average over heads.
        return S.mean(dim=1).squeeze(0).numpy()


cosines_per_layer = []
for li in range(model.n_layers):
    S = measured_gram_at_layer(model, X_all, y_c, n_ctx=X_c.shape[0], layer_idx=li)
    S_sym = (S + S.T) / 2
    # Cosine in Frobenius inner product.
    num = float((S_sym * oracle_gram_S).sum())
    den = float(np.sqrt((S_sym ** 2).sum() * (oracle_gram_S ** 2).sum()) + 1e-12)
    cosines_per_layer.append(num / den)

print('per-layer Frobenius cosine of S_sym to oracle Gram:')
for li, c in enumerate(cosines_per_layer):
    print(f'  layer {li}: {c:.3f}')

fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))
ax.bar(np.arange(len(cosines_per_layer)), cosines_per_layer, color='C0', alpha=0.85)
ax.axhline(0, color='gray', alpha=0.5)
ax.set_xlabel('layer'); ax.set_ylabel('Frobenius cosine')
ax.set_title('Figure 16.2: Cosine recovery of $S_\\mathrm{sym}$ to oracle Gram')
ax.set_xticks(np.arange(len(cosines_per_layer)))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(f'{FIGURES_DIR}/fig_16_02_cosine_recovery.pdf', bbox_inches='tight')
plt.show()
